In [10]:
from torch.utils.data import Dataset, DataLoader
import torch
from tqdm import tqdm

class HMDataset(Dataset):
    def __init__(self, x):
        self.x = x.reset_index(drop=True)

    def __len__(self):
        return self.x.shape[0]

    def __getitem__(self, index):

        row = self.x.iloc[index]
        user = torch.tensor(row.customer_id)
        item = torch.tensor(row.product_code)
        target = torch.tensor([row.bought]).float()

        return user,item,target


In [12]:
def get_optimizer(net):
    optimizer = torch.optim.SGD(net.parameters(), lr=0.01)

    return optimizer

In [14]:
import torch.nn as nn
import torch.nn.functional as F

class HMModel(nn.Module):
    def __init__(self,n_users,n_items):
        super(HMModel, self).__init__()
        torch.manual_seed(1)

        self.user_emb = nn.Embedding(n_users, embedding_dim=30).float()
        self.article_emb = nn.Embedding(n_items, embedding_dim=30).float()
        self.user_bias = nn.Embedding(n_users, embedding_dim=1).float()
        self.article_bias = nn.Embedding(n_items, embedding_dim=1).float()
        self.age_vector = torch.zeros(n_items)#Record updating times


    def forward(self, inputs):
        user_index, item_index = inputs[0], inputs[1]
        u_e = self.user_emb(user_index)
        i_e = self.article_emb(item_index)
        u_b = self.user_bias(user_index)
        i_b = self.article_bias(item_index)
        x = (u_e * i_e).sum(1) + u_b + i_b

        return x

In [16]:
#Models Initialization
model_dict = {}
for x in tqdm(range(n_users)):
    model_dict["model{0}".format(x)] = HMModel(n_users,n_items)

NameError: name 'n_users' is not defined

In [ ]:
import sys

def read_data(data):
    return tuple(d for d in data[:-1]), data[-1]

def validate(model_dict, val_loader):

    tbar = val_loader
    criterion = nn.MSELoss()
    loss_list = []
    for i in range(len(model_dict)):
        model_dict["model{0}".format(i)].eval()

    with torch.no_grad():
        for idx, data in enumerate(tbar):
            inputs, target = read_data(data)
            logits = model_dict["model{0}".format(int(inputs[0]))](inputs)
            loss = criterion(logits, target)

            loss_list.append(loss.detach().cpu().item())
        avg_loss = np.sqrt(np.mean(loss_list))

    return avg_loss

val_dataset = HMDataset(val_df)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False)

In [ ]:
val_loss = []
optimizer_dict = {}
def train(model_dict, train_loader, epochs):

    criterion = nn.MSELoss()
    val_map = validate(model_dict, val_loader)
    val_loss.append(val_map)
    for i in tqdm(range(n_users)):
        optimizer_dict["model{0}".format(i)] = get_optimizer(model_dict["model{0}".format(i)])

    for e in range(epochs):
        tbar = tqdm(train_loader, file=sys.stdout)
        for i in range(len(model_dict)):
            model_dict["model{0}".format(i)].train()
        loss_list = []
        count = 0
        neighbors = []
        for idx, data in enumerate(tbar):
            inputs, target = read_data(data)
            article_index = int(inputs[1])
            customer_index = int(inputs[0])
            optimizer = optimizer_dict["model{0}".format(int(inputs[0]))]
            optimizer.zero_grad()
            neighbors = list(Adjacency_List[customer_index])

            for neighbor in neighbors:

                # Get both information
                n_age_vector = model_dict["model{0}".format(neighbor)].age_vector[article_index]
                if n_age_vector == 0: continue
                n_item_embedding = model_dict["model{0}".format(neighbor)].article_emb(torch.tensor([article_index]))
                current_item_embedding = model_dict["model{0}".format(int(inputs[0]))].article_emb(torch.tensor([article_index]))
                current_age_vector = model_dict["model{0}".format(int(inputs[0]))].age_vector[article_index]
                current_c = model_dict["model{0}".format(int(inputs[0]))].article_bias(torch.tensor([article_index]))
                n_c = model_dict["model{0}".format(neighbor)].article_bias(torch.tensor([article_index]))

                # Merge
                weight_current_age_vector = current_age_vector/(current_age_vector+n_age_vector)
                weight_n_age_vector = n_age_vector/(current_age_vector+n_age_vector)
                model_dict["model{0}".format(int(inputs[0]))].age_vector[article_index] = torch.max(torch.tensor([current_age_vector,n_age_vector]))

                modified_weight = model_dict["model{0}".format(int(inputs[0]))].article_emb.weight.clone()
                modified_weight[article_index] = nn.Parameter(torch.add(torch.multiply(current_item_embedding,weight_current_age_vector),torch.multiply(n_item_embedding,weight_n_age_vector)))
                model_dict["model{0}".format(int(inputs[0]))].article_emb.weight = nn.Parameter(modified_weight)

                modified_weight = model_dict["model{0}".format(int(inputs[0]))].article_bias.weight.clone()
                modified_weight[article_index] = nn.Parameter(torch.add(torch.multiply(current_c,weight_current_age_vector),torch.multiply(n_c,weight_n_age_vector)))
                model_dict["model{0}".format(int(inputs[0]))].article_bias.weight = nn.Parameter(modified_weight)


            model_dict["model{0}".format(int(inputs[0]))].age_vector[article_index] += 1
            logits = model_dict["model{0}".format(int(inputs[0]))](inputs)
            loss = criterion(logits, target)
            loss.backward()
            optimizer.step()

            loss_list.append(loss.detach().cpu().item())
            avg_loss = np.mean(loss_list)


        val_map = validate(model_dict, val_loader)
        val_loss.append(val_map)
        log_text = f"Epoch {e+1}\nTrain Loss: {np.sqrt(avg_loss)}\nValidation Loss: {val_map}\n"
        print(log_text)




train_dataset = HMDataset(train_df)
train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True)
train(model_dict, train_loader,epochs=10)

In [ ]:
val_loss #Evaluation